# Lab | Agent & Vector store

**Change the state union dataset and replicate this lab by updating the prompts accordingly.**

One such dataset is the [sonnets.txt](https://github.com/martin-gorner/tensorflow-rnn-shakespeare/blob/master/shakespeare/sonnets.txt) dataset or any other data of your choice from the same git.

# Combine agents and vector stores

This notebook covers how to combine agents and vector stores. The use case for this is that you've ingested your data into a vector store and want to interact with it in an agentic manner.

The recommended method for doing so is to create a `RetrievalQA` and then use that as a tool in the overall agent. Let's take a look at doing this below. You can do this with multiple different vector DBs, and use the agent as a way to route between them. There are two different ways of doing this - you can either let the agent use the vector stores as normal tools, or you can set `return_direct=True` to really just use the agent as a router.

## Create the vector store

In [4]:
!pip install chromadb langchain==0.1.20 langchain-community==0.0.38 langchain-openai==0.1.7

INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
  Attempting uninstall: build
    Found existing installation: build 1.5.0
    Uninstalling build-1.5.0:
      Successfully uninstalled build-1.5.0


In [6]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [7]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [37]:
llm = OpenAI(temperature=0)

In [38]:
from pathlib import Path

relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break
doc_path = str(Path(*relevant_parts) / "state_of_the_union.txt")

In [39]:
loader = TextLoader(doc_path)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()
docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

In [11]:
state_of_union = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

In [12]:
from langchain_community.document_loaders import WebBaseLoader

In [13]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [14]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

## Create the Agent

In [15]:
# Import things that are needed generically

from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

In [40]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for answering questions about Shakespeare's sonnets, including themes, love, beauty, time, aging, mortality, metaphors, and poetic language. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [17]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(


In [18]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
 I should use the State of Union QA System to answer this question
Action: State of Union QA System
Action Input: What did biden say about ketanji brown jackson in the state of the union address?
Observation:  Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, praising her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.
Thought: I now know the final answer
Final Answer: Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, praising her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.

> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': "Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, praising her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}

In [19]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 You should always think about the differences between ruff and flake8
Action: Ruff QA System
Action Input: "Why use ruff over flake8?"
Observation: 
There are a few reasons why someone might choose to use Ruff over Flake8:

1. Larger rule set: Ruff implements over 800 rules, while Flake8 only implements around 200. This means that Ruff can catch more potential issues in your code.

2. Better compatibility with other tools: Ruff is designed to work well with other tools like Black, isort, and type checkers like Mypy. This means that you can use Ruff alongside these tools to get more comprehensive feedback on your code.

3. Automatic fixing of lint violations: Unlike Flake8, Ruff is capable of automatically fixing its own lint violations. This can save you time and effort when fixing issues in your code.

4. Native implementation of popular Flake8 plugins: Ruff re-implements some of the most popular Flake8 plugins natively, which means you don't h

{'input': 'Why use ruff over flake8?',
 'output': 'Ruff offers a larger rule set, better compatibility with other tools, automatic fixing of lint violations, and native implementation of popular Flake8 plugins, making it a more comprehensive and user-friendly option compared to Flake8.'}

## Use the Agent solely as a router

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [34]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for answering questions about Shakespeare's sonnets, including themes, love, beauty, time, aging, mortality, metaphors, and poetic language. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [24]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [22]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)



> Entering new AgentExecutor chain...
 I should use the State of Union QA System to answer this question
Action: State of Union QA System
Action Input: What did biden say about ketanji brown jackson in the state of the union address?
Observation:  Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, praising her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': " Biden mentioned that he nominated Circuit Court of Appeals Judge Ketanji Brown Jackson to serve on the United States Supreme Court, praising her as one of the nation's top legal minds who will continue Justice Breyer's legacy of excellence."}

In [23]:
agent.invoke("Why use ruff over flake8?")



> Entering new AgentExecutor chain...
 You should always think about the differences between ruff and flake8
Action: Ruff QA System
Action Input: Why use ruff over flake8?
Observation: 
There are a few reasons why someone might choose to use Ruff over Flake8:

1. Larger rule set: Ruff implements over 800 rules, while Flake8 only implements around 200. This means that Ruff can catch more potential issues in your code.

2. Better compatibility with other tools: Ruff is designed to work well with other tools like Black, isort, and yesqa, while Flake8 may have conflicts with these tools.

3. Automatic fixing of lint violations: Ruff is capable of automatically fixing its own lint violations, while Flake8 does not have this capability.

4. Native implementation of popular Flake8 plugins: Ruff natively implements some of the most popular Flake8 plugins, making it more efficient and less prone to conflicts.

5. Can be used as a formatter: Ruff can be used as a formatter, making it a more ve

{'input': 'Why use ruff over flake8?',
 'output': '\nThere are a few reasons why someone might choose to use Ruff over Flake8:\n\n1. Larger rule set: Ruff implements over 800 rules, while Flake8 only implements around 200. This means that Ruff can catch more potential issues in your code.\n\n2. Better compatibility with other tools: Ruff is designed to work well with other tools like Black, isort, and yesqa, while Flake8 may have conflicts with these tools.\n\n3. Automatic fixing of lint violations: Ruff is capable of automatically fixing its own lint violations, while Flake8 does not have this capability.\n\n4. Native implementation of popular Flake8 plugins: Ruff natively implements some of the most popular Flake8 plugins, making it more efficient and less prone to conflicts.\n\n5. Can be used as a formatter: Ruff can be used as a formatter, making it a more versatile tool than Flake8.\n\nUltimately, the choice between Ruff and Flake8 will depend on the specific needs and preferences

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [35]:
tools = [
    Tool(
        name="Sonnets QA System",
        func=sonnets.run,
        description="useful for answering questions about Shakespeare's sonnets, including themes, love, beauty, time, aging, mortality, metaphors, and poetic language. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [43]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [27]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?"
)



> Entering new AgentExecutor chain...
 I should use the Ruff QA System to answer the first question and the State of Union QA System to answer the second question.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks?
Observation:  Ruff uses nbQA, a tool for running linters and code formatters over Jupyter Notebooks.
Thought: I should now use the State of Union QA System to answer the second question.
Action: State of Union QA System
Action Input: Did the president mention that tool in the state of the union?
Observation: 
No, the president did not mention a specific tool in the state of the union.
Thought: I now know the final answer.
Final Answer: No, the president did not mention a specific tool in the state of the union.

> Finished chain.


{'input': 'What tool does ruff use to run over Jupyter Notebooks? Did the president mention that tool in the state of the union?',
 'output': 'No, the president did not mention a specific tool in the state of the union.'}

In [28]:
relevant_parts = []
for p in Path(".").absolute().parts:
    relevant_parts.append(p)
    if relevant_parts[-3:] == ["langchain", "docs", "modules"]:
        break


sonnet_doc_path = str(Path(*relevant_parts) / "sonnets.txt")

In [29]:
loader = TextLoader(sonnet_doc_path)
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings()

docsearch = Chroma.from_documents(
    texts,
    embeddings,
    collection_name="sonnets"
    )

In [41]:
sonnets = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever()
)

In [44]:
agent.invoke("How does Shakespeare describe love and beauty in the sonnets?")



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System to answer this question
Action: Sonnets QA System
Action Input: How does Shakespeare describe love and beauty in the sonnets?
Observation:  In the sonnets, Shakespeare often describes love and beauty as timeless and eternal, using imagery of nature and classical references to convey their enduring qualities. He also emphasizes the idea of true love being constant and unchanging, even as physical beauty fades. Additionally, he often contrasts true love with superficial or artificial forms of love and beauty.
Thought: I now know the final answer
Final Answer: In the sonnets, Shakespeare often describes love and beauty as timeless and eternal, using imagery of nature and classical references to convey their enduring qualities. He also emphasizes the idea of true love being constant and unchanging, even as physical beauty fades. Additionally, he often contrasts true love with superficial or artificial forms of love

{'input': 'How does Shakespeare describe love and beauty in the sonnets?',
 'output': 'In the sonnets, Shakespeare often describes love and beauty as timeless and eternal, using imagery of nature and classical references to convey their enduring qualities. He also emphasizes the idea of true love being constant and unchanging, even as physical beauty fades. Additionally, he often contrasts true love with superficial or artificial forms of love and beauty.'}

In [45]:
agent.invoke("What role does time play in Shakespeare's sonnets?")



> Entering new AgentExecutor chain...
 Time is a recurring theme in Shakespeare's sonnets, so the Sonnets QA System should be able to provide some insight.
Action: Sonnets QA System
Action Input: "What is the role of time in Shakespeare's sonnets?"
Observation:  In Shakespeare's sonnets, time is portrayed as a powerful force that ultimately leads to decay and death. The speaker often reflects on the fleeting nature of time and how it affects youth, beauty, and love. Time is seen as an enemy, constantly working against the speaker's desire to preserve and immortalize the subject of the sonnets. However, the speaker also acknowledges that time is a natural and necessary part of life, and ultimately accepts its inevitability.
Thought: This is a complex and nuanced answer, but I think I have a better understanding of the role of time in Shakespeare's sonnets now.
Final Answer: In Shakespeare's sonnets, time is portrayed as a powerful force that ultimately leads to decay and death, but al

{'input': "What role does time play in Shakespeare's sonnets?",
 'output': "In Shakespeare's sonnets, time is portrayed as a powerful force that ultimately leads to decay and death, but also a natural and necessary part of life that the speaker must come to terms with."}

In [47]:
#Multi-Hop
agent.invoke(
    "What does Shakespeare say about time in the sonnets, and does Ruff mention tools related to timing or performance?"
)



> Entering new AgentExecutor chain...
 I should use the Sonnets QA System to answer the first part of the question and the Ruff QA System to answer the second part.
Action: Sonnets QA System
Action Input: What themes does Shakespeare explore in his sonnets?
Observation:  Shakespeare explores themes of mortality, the fleeting nature of beauty, the power of love, and the struggle against time and decay in his sonnets.
Thought: I should now use the Ruff QA System to answer the second part of the question.
Action: Ruff QA System
Action Input: Does Ruff have any tools related to timing or performance?
Observation:  No, Ruff does not currently have any tools related to timing or performance.
Thought: I now know the final answer.
Final Answer: Shakespeare explores themes of mortality, the fleeting nature of beauty, the power of love, and the struggle against time and decay in his sonnets. Ruff does not currently have any tools related to timing or performance.

> Finished chain.


{'input': 'What does Shakespeare say about time in the sonnets, and does Ruff mention tools related to timing or performance?',
 'output': 'Shakespeare explores themes of mortality, the fleeting nature of beauty, the power of love, and the struggle against time and decay in his sonnets. Ruff does not currently have any tools related to timing or performance.'}

Changed the dataset, added new questions, including one multi-hop.